In [8]:
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool

llm = init_chat_model("openai:gpt-5.4-mini")

In [9]:
class State(MessagesState):
    custom_stuff: str


graph_builder = StateGraph(State)

In [10]:
@tool
def get_weather(city: str):
    """Gets weather in city"""
    return f"The weather in {city} is sunny."


llm_with_tools = llm.bind_tools(tools=[get_weather])


def chatbot(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

In [ ]:
tool_node = ToolNode(
    tools=[get_weather],
)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)


graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

graph = graph_builder.compile()

graph.invoke(
    {
        "messages": [
            {"role": "user", "content": "안녕"},
        ]
    }
)

{'messages': [HumanMessage(content='hello', additional_kwargs={}, response_metadata={}, id='c024a171-4f59-4375-86bd-f97197d4d436'),
  AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 124, 'total_tokens': 136, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DWxNbECjgUQx9xq9K5o8ZncpCcVKE', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dae61-8ce8-7292-9a75-8619e430ed9c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 124, 'output_tokens': 12, 'total_tokens': 136, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'